Den här koden laddar in ett dataset med fordonsdata och städar det genom att rensa bort ofullständiga rader samt kolumnen med bilnamn, eftersom alla unika namn skulle skapa för mycket brus i analysen. Därefter omvandlas textinformationen om bilarnas ursprung till ett numeriskt format som en algoritm faktiskt kan förstå. Slutligen delas datan upp i två delar för att först träna en linjär regressionsmodell på majoriteten av informationen, och därefter utvärdera hur väl modellen lyckas förutsäga bränsleförbrukningen på den helt osedda testdatan.

In [2]:
# Importera nödvändiga bibliotek
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [3]:
# a) Läs in mpg-datasetet från seaborn
df = sns.load_dataset("mpg")
print(f"Ursprunglig storlek: {df.shape}")

# b) Droppa rader med saknade värden (NaN)
df = df.dropna()

# c) Droppa kolumnen 'name' eftersom dess 305 unika värden inte lämpar sig för vår modell
df = df.drop(columns=['name'])

print(f"Storlek efter att rader med saknade värden och 'name'-kolumnen tagits bort: {df.shape}")

Ursprunglig storlek: (398, 9)
Storlek efter att rader med saknade värden och 'name'-kolumnen tagits bort: (392, 8)


In [4]:
# d) Utför dummy-variable-encoding på 'origin'
# drop_first=True innebär att vi bara skapar 2 kolumner för 3 kategorier (förhindrar the dummy variable trap)
# dtype=int ser till att vi får 1:or och 0:or istället för True/False
df = pd.get_dummies(df, columns=['origin'], drop_first=True, dtype=int)

# Ta en snabb titt på de 5 första raderna för att verifiera att vi fått våra dummy-kolumner
print(df.head())

    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year  origin_japan  origin_usa  
0          70             0           1  
1          70             0           1  
2          70             0           1  
3          70             0           1  
4          70             0           1  


In [ ]:
# e) Dela upp datasetet i X (oberoende variabler) och y (beroende variabel)
X = df.drop(columns=['mpg'])
y = df['mpg']

# f) Dela upp datasetet i tränings- och testset ( 80% träning, 20% test)
# random_state=42 används för att vi ska få samma uppdelning varje gång koden körs
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Träningsdatans storlek: {X_train.shape}")
print(f"Testdatans storlek: {X_test.shape}")

Träningsdatans storlek: (313, 8)
Testdatans storlek: (79, 8)


In [6]:
# g) Instantiera och träna den linjära regressionsmodellen på träningsdatan
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Gör prediktioner på testdatan
y_pred = lin_reg.predict(X_test)

# Utvärdera modellen genom att beräkna RMSE (Root Mean Squared Error)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"Modellens RMSE på testdatan är: {rmse:.2f} mpg")

Modellens RMSE på testdatan är: 3.26 mpg
